# Práctica 5 — Tarea 1: Espejo de emociones

**Escenario de aplicación.** Un *espejo emocional* para la webcam: detecta la cara, reconoce la **emoción** con un modelo entrenado por nosotros y **reacciona** con efectos anclados a los landmarks faciales (lágrimas si estás triste, rubor y destellos si estás contento, vapor si te enfadas...). Incluye un **modo juego**, *"Imita la emoción"*: el sistema pide una emoción y suma un punto cuando el usuario la mantiene durante un segundo.

Posibles usos: entrenamiento de la expresión emocional (por ejemplo, en terapias con niños), juegos interactivos o *feedback* al ensayar una presentación.

**Modelo biométrico propio.** Se entrena un clasificador de **7 emociones** con el dataset **FER-2013** y se comparan dos enfoques:

* **Clásico:** descriptores HOG + LBP con una SVM lineal.
* **Red neuronal con *transfer learning*:** MobileNetV2 preentrenada en ImageNet y ajustada a FER-2013.

| Parte | Contenido |
|---|---|
| 1 | Carga, exploración y preprocesado del dataset (incluida la alineación de la cara con landmarks) |
| 2 | Entrenamiento, evaluación y comparativa de los dos modelos (y extra: comparativa con `DeepFace.analyze`) |
| 3 | Prototipo en tiempo real con la webcam |

Instalaciones previas

In [ ]:
# Además del entorno VC_P5 del enunciado, para leer el dataset (formato parquet) hace falta pyarrow
#%pip install pyarrow

Paquetes necesarios

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # menos mensajes de TensorFlow

import io
import json
import time
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import mediapipe as mp

CONFIGURACIÓN (rutas y parámetros)

In [ ]:
# Rutas (relativas a la carpeta TAREAS/)
CARPETA_MODELOS    = '../modelos'
DATASET_DIR        = '../data/FER2013'          # train.parquet, valid.parquet, test.parquet
CARPETA_TEST_IMG   = '../img/test'
CARPETA_RESULTADOS = '../resultados/tarea1'
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

FACE_LANDMARKER = os.path.join(CARPETA_MODELOS, 'face_landmarker.task')
MODELO_CNN      = os.path.join(CARPETA_MODELOS, 'emociones_mobilenetv2.keras')
MODELO_CLASICO  = os.path.join(CARPETA_MODELOS, 'emociones_hog_lbp_svm.joblib')

# Clases de FER-2013, en el orden de sus etiquetas (0..6). Sin tildes porque cv2.putText no las dibuja.
EMOCIONES = ['Enfado', 'Asco', 'Miedo', 'Alegria', 'Tristeza', 'Sorpresa', 'Neutral']
N_CLASES  = len(EMOCIONES)
TAM_CARA  = 48           # tamaño de las imágenes de FER-2013
SEMILLA   = 42
np.random.seed(SEMILLA)

# Colores (BGR) por emoción, usados en las gráficas y en el prototipo
COLORES = {'Enfado': (40, 40, 230), 'Asco': (40, 160, 40), 'Miedo': (180, 60, 160), 'Alegria': (0, 210, 255),
           'Tristeza': (230, 140, 30), 'Sorpresa': (0, 140, 255), 'Neutral': (190, 190, 190)}
COLORES_PLT = [tuple(c / 255 for c in COLORES[e][::-1]) for e in EMOCIONES]

---
# PARTE 1: DATASET

PASO 1.1: ELECCIÓN DEL DATASET

Se usa **FER-2013** (*Facial Expression Recognition*), publicado para el reto de reconocimiento de expresiones del ICML 2013 y uno de los conjuntos de referencia para esta tarea:

* **35 887 caras** en escala de grises de **48×48 píxeles**, ya recortadas y aproximadamente centradas.
* **7 emociones:** enfado, asco, miedo, alegría, tristeza, sorpresa y neutral.
* **Particiones oficiales:** *Training* (28 709), *PublicTest* (3589, usada como validación) y *PrivateTest* (3589, usada como test).
* Copia utilizada: [AutumnQiu/fer2013 en HuggingFace](https://huggingface.co/datasets/AutumnQiu/fer2013), en formato parquet, con las particiones originales. Se guarda en `PRACTICA_5/data/FER2013/` (ignorado por git).

**Por qué este dataset:** es grande, tiene las caras ya recortadas (como las que produce nuestro detector), está etiquetado por emoción y sus imágenes, de baja resolución y tomadas en condiciones reales, se parecen a lo que capta una webcam. Su principal dificultad es conocida: el **desbalance de clases** (hay 16 veces más caras alegres que de asco) y la **ambigüedad entre emociones** (la precisión humana en FER-2013 se estima en torno al 65 %).

In [ ]:
for nombre in ['train', 'valid', 'test']:
    ruta = os.path.join(DATASET_DIR, f'{nombre}.parquet')
    print(f'{nombre:5s} -> {ruta}  ({os.path.getsize(ruta) / 1e6:.1f} MB)')

PASO 1.2: CARGA Y EXPLORACIÓN

In [ ]:
def cargar_split(nombre):
    """Lee un split del parquet y devuelve las caras (N, 48, 48) en grises uint8 y sus etiquetas."""
    df = pd.read_parquet(os.path.join(DATASET_DIR, f'{nombre}.parquet'))
    X = np.stack([np.array(Image.open(io.BytesIO(img['bytes'])).convert('L')) for img in df['image']])
    y = df['label'].to_numpy().astype(np.int64)
    return X, y


t0 = time.time()
X_train, y_train = cargar_split('train')
X_val,   y_val   = cargar_split('valid')
X_test,  y_test  = cargar_split('test')
print(f'Cargado en {time.time() - t0:.1f} s')
print('train:', X_train.shape, '| val:', X_val.shape, '| test:', X_test.shape, '| dtype:', X_train.dtype)

In [ ]:
# Ejemplos de cada emoción
n_ej = 8
plt.figure(figsize=(14, 13))
for c, emocion in enumerate(EMOCIONES):
    idx = np.random.choice(np.where(y_train == c)[0], n_ej, replace=False)
    for j, i in enumerate(idx):
        plt.subplot(N_CLASES, n_ej, c * n_ej + j + 1)
        plt.imshow(X_train[i], cmap='gray')
        plt.axis('off')
        if j == 0:
            plt.title(emocion, loc='left', fontsize=11)
plt.suptitle('FER-2013: ejemplos de cada emoción', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'ejemplos_dataset.png'), dpi=100)
plt.show()

In [ ]:
# Número de muestras por clase y split
conteo = pd.DataFrame({split: np.bincount(y, minlength=N_CLASES)
                       for split, y in [('train', y_train), ('val', y_val), ('test', y_test)]}, index=EMOCIONES)
display(conteo.assign(porcentaje_train=(conteo.train / conteo.train.sum() * 100).round(1)))

ax = conteo.plot.bar(figsize=(10, 4), color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_ylabel('Nº de imágenes'); ax.set_title('Distribución de clases en FER-2013')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'distribucion_clases.png'), dpi=110)
plt.show()

print(f'Desbalance: Alegria tiene {conteo.train.max() / conteo.train.min():.1f} veces más imágenes que Asco')

In [ ]:
# Cara media de cada emoción: resume qué rasgos distinguen cada clase
plt.figure(figsize=(14, 2.6))
for c, emocion in enumerate(EMOCIONES):
    plt.subplot(1, N_CLASES, c + 1)
    plt.imshow(X_train[y_train == c].mean(axis=0), cmap='gray')
    plt.title(emocion); plt.axis('off')
plt.suptitle('Cara media por emoción')
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'cara_media.png'), dpi=110)
plt.show()

PASO 1.3: PREPROCESADO DE LA CARA (detección + landmarks + alineado)

Las imágenes de FER-2013 ya son caras recortadas de 48×48 en grises, así que para entrenar basta con normalizar los valores (se hace dentro de cada modelo). Lo importante es que **en el prototipo la cara llegue al modelo con el mismo aspecto que en el dataset**. Para ello, con los **478 landmarks de MediaPipe Face Landmarker**:

1. Se calcula el centro de cada ojo (puntos 33-133 y 362-263) y el **ángulo** de la línea que los une.
2. Se **rota** la imagen alrededor del punto medio de los ojos para que queden horizontales (cara alineada, como en FER-2013).
3. Se toma el **óvalo facial** (36 landmarks del contorno) ya rotado, se recorta un **cuadrado** centrado en él con un 10 % de margen y se pasa a grises y a 48×48.

In [ ]:
# Índices de landmarks de MediaPipe Face Mesh
OJO_IZQ   = [33, 133]     # esquinas exterior e interior del ojo izquierdo (de la imagen)
OJO_DER   = [362, 263]    # esquinas interior y exterior del ojo derecho
OVALO     = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377,
             152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
FRENTE, BARBILLA = 10, 152
MEJILLA_IZQ, MEJILLA_DER = 50, 280
PARPADO_IZQ, PARPADO_DER = 145, 374   # párpado inferior (de donde caen las lágrimas)


def landmarks_a_pixeles(resultado, ancho, alto):
    """Convierte el resultado de MediaPipe en una lista de arrays (478, 2) en píxeles, uno por cara."""
    return [np.array([(p.x * ancho, p.y * alto) for p in cara], dtype=np.float32)
            for cara in resultado.face_landmarks]


def recorte_cuadrado(img, cx, cy, lado):
    """Recorta un cuadrado centrado en (cx, cy); si se sale de la imagen, rellena replicando el borde."""
    alto, ancho = img.shape[:2]
    lado = int(round(lado))
    x1, y1 = int(round(cx - lado / 2)), int(round(cy - lado / 2))
    x2, y2 = x1 + lado, y1 + lado
    pad = max(0, -x1, -y1, x2 - ancho, y2 - alto)
    if pad:
        img = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
        x1, y1, x2, y2 = x1 + pad, y1 + pad, x2 + pad, y2 + pad
    return img[y1:y2, x1:x2]


def recortar_cara_alineada(frame_bgr, puntos, tam=TAM_CARA, margen=0.10):
    """Alinea la cara con los ojos y devuelve el recorte en grises (tam x tam), como las caras de FER-2013."""
    ojo_i = puntos[OJO_IZQ].mean(axis=0)
    ojo_d = puntos[OJO_DER].mean(axis=0)
    angulo = np.degrees(np.arctan2(ojo_d[1] - ojo_i[1], ojo_d[0] - ojo_i[0]))
    centro = (ojo_i + ojo_d) / 2
    M = cv2.getRotationMatrix2D((float(centro[0]), float(centro[1])), angulo, 1.0)
    alto, ancho = frame_bgr.shape[:2]
    rotada = cv2.warpAffine(frame_bgr, M, (ancho, alto), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

    ovalo = puntos[OVALO] @ M[:, :2].T + M[:, 2]          # óvalo facial en la imagen rotada
    (x1, y1), (x2, y2) = ovalo.min(axis=0), ovalo.max(axis=0)
    lado = max(x2 - x1, y2 - y1) * (1 + margen)
    cara = recorte_cuadrado(rotada, (x1 + x2) / 2, (y1 + y2) / 2, lado)
    gris = cv2.cvtColor(cara, cv2.COLOR_BGR2GRAY)
    return cv2.resize(gris, (tam, tam), interpolation=cv2.INTER_AREA)

In [ ]:
# Prueba del preprocesado con las imágenes de ../img/test
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

opciones_imagen = FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=FACE_LANDMARKER),
                                        running_mode=VisionRunningMode.IMAGE, num_faces=1)
imagenes_test = sorted(f for f in os.listdir(CARPETA_TEST_IMG) if f.lower().endswith(('.jpg', '.png')))

plt.figure(figsize=(12, 4 * len(imagenes_test)))
with FaceLandmarker.create_from_options(opciones_imagen) as detector_img:
    for k, nombre in enumerate(imagenes_test):
        img = cv2.imread(os.path.join(CARPETA_TEST_IMG, nombre))
        res = detector_img.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
        caras = landmarks_a_pixeles(res, img.shape[1], img.shape[0])
        vis = img.copy()
        for pts in caras:
            cv2.polylines(vis, [pts[OVALO].astype(np.int32)], True, (0, 255, 0), 2)
            for i in OJO_IZQ + OJO_DER:
                cv2.circle(vis, tuple(pts[i].astype(int)), 4, (0, 0, 255), -1)
        plt.subplot(len(imagenes_test), 2, 2 * k + 1)
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.title(f'{nombre}: óvalo facial y ojos'); plt.axis('off')
        plt.subplot(len(imagenes_test), 2, 2 * k + 2)
        if caras:
            plt.imshow(recortar_cara_alineada(img, caras[0]), cmap='gray'); plt.title('Recorte alineado 48x48')
        else:
            plt.title('Sin cara detectada')
        plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'preprocesado_landmarks.png'), dpi=90)
plt.show()

PASO 1.4: DIVISIÓN TRAIN / VALIDACIÓN / TEST

Se mantienen las **particiones oficiales** de FER-2013, que permiten comparar con otros trabajos:

* **Train** (28 709): para entrenar.
* **Validación** (PublicTest, 3589): para elegir hiperparámetros y el mejor modelo (parada temprana).
* **Test** (PrivateTest, 3589): solo para la evaluación final.

Para compensar el **desbalance** se usan **pesos por clase** en el entrenamiento: las clases con pocas imágenes (sobre todo *Asco*) pesan más en la función de pérdida. Los pesos *balanced* se suavizan con una raíz cuadrada para no sobrecompensar *Asco*, que tiene solo 436 ejemplos.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

pesos = compute_class_weight('balanced', classes=np.arange(N_CLASES), y=y_train)
PESOS_CLASE = {c: float(np.sqrt(p)) for c, p in enumerate(pesos)}
display(pd.DataFrame({'muestras_train': np.bincount(y_train), 'peso_balanced': pesos.round(2),
                      'peso_usado (raíz)': [round(PESOS_CLASE[c], 2) for c in range(N_CLASES)]}, index=EMOCIONES))

---
# PARTE 2: ENTRENAMIENTO DE LOS MODELOS

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

RESULTADOS = {}     # métricas de cada modelo para la comparativa


def evaluar(nombre, y_real, y_pred, guardar=True):
    """Calcula y guarda las métricas de un modelo y muestra el informe por clase."""
    m = {'accuracy': accuracy_score(y_real, y_pred),
         'f1_macro': f1_score(y_real, y_pred, average='macro'),
         'f1_ponderado': f1_score(y_real, y_pred, average='weighted'),
         'f1_por_clase': f1_score(y_real, y_pred, average=None, labels=range(N_CLASES)),
         'matriz': confusion_matrix(y_real, y_pred, labels=range(N_CLASES))}
    if guardar:
        RESULTADOS.setdefault(nombre, {}).update(m)
    print(f'{nombre}: accuracy {m["accuracy"]:.3f} | F1 macro {m["f1_macro"]:.3f} | F1 ponderado {m["f1_ponderado"]:.3f}')
    print(classification_report(y_real, y_pred, labels=range(N_CLASES), target_names=EMOCIONES, digits=3, zero_division=0))
    return m


def dibujar_matriz(ax, matriz, titulo):
    """Matriz de confusión normalizada por filas (recall de cada clase en la diagonal)."""
    norm = matriz / matriz.sum(axis=1, keepdims=True)
    ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(N_CLASES):
        for j in range(N_CLASES):
            ax.text(j, i, f'{norm[i, j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if norm[i, j] > 0.5 else 'black')
    ax.set_xticks(range(N_CLASES), EMOCIONES, rotation=45, ha='right')
    ax.set_yticks(range(N_CLASES), EMOCIONES)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real'); ax.set_title(titulo)

PASO 2.1: MODELO CLÁSICO (HOG + LBP + SVM lineal)

Cada cara se describe con dos descriptores locales vistos en la práctica:

* **HOG** (histogramas de gradientes orientados): celdas de 6×6 píxeles, 9 orientaciones y bloques de 2×2 celdas. Captura la **forma** de cejas, ojos y boca. Da 1764 valores.
* **LBP** uniforme (P=8, R=1): textura local, en una rejilla de 4×4 regiones con un histograma de 10 valores cada una. Da 160 valores.

Las 1924 características se estandarizan y se clasifican con una **SVM lineal** con pesos por clase. El parámetro de regularización `C` se elige con el conjunto de validación.

In [ ]:
from skimage.feature import hog, local_binary_pattern
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def caracteristicas_clasicas(caras):
    """HOG + histogramas LBP por regiones para un array de caras (N, 48, 48) uint8."""
    salida = []
    for cara in caras:
        f_hog = hog(cara, orientations=9, pixels_per_cell=(6, 6), cells_per_block=(2, 2),
                    block_norm='L2-Hys', feature_vector=True)
        lbp = local_binary_pattern(cara, P=8, R=1, method='uniform')      # valores 0..9
        hist = []
        for i in range(4):
            for j in range(4):
                region = lbp[i * 12:(i + 1) * 12, j * 12:(j + 1) * 12]
                h, _ = np.histogram(region, bins=10, range=(0, 10))
                hist.append(h / h.sum())
        salida.append(np.concatenate([f_hog, np.concatenate(hist)]))
    return np.asarray(salida, dtype=np.float32)


t0 = time.time()
F_train = caracteristicas_clasicas(X_train)
F_val   = caracteristicas_clasicas(X_val)
F_test  = caracteristicas_clasicas(X_test)
print(f'Características: {F_train.shape[1]} por cara, extraídas en {time.time() - t0:.0f} s')

In [ ]:
# Elección de C con el conjunto de validación (F1 macro, que tiene en cuenta todas las clases por igual)
mejor_c, mejor_f1 = None, -1
for C in [0.001, 0.003, 0.01, 0.03]:
    t0 = time.time()
    m = make_pipeline(StandardScaler(), LinearSVC(C=C, class_weight=PESOS_CLASE, max_iter=5000, random_state=SEMILLA))
    m.fit(F_train, y_train)
    f1v = f1_score(y_val, m.predict(F_val), average='macro')
    print(f'C={C:<6} F1 macro val = {f1v:.3f}  ({time.time() - t0:.0f} s)')
    if f1v > mejor_f1:
        mejor_c, mejor_f1 = C, f1v

t0 = time.time()
modelo_clasico = make_pipeline(StandardScaler(), LinearSVC(C=mejor_c, class_weight=PESOS_CLASE,
                                                           max_iter=5000, random_state=SEMILLA))
modelo_clasico.fit(F_train, y_train)
TIEMPO_ENTRENO_CLASICO = time.time() - t0
print(f'\nMejor C = {mejor_c}. Entrenamiento final: {TIEMPO_ENTRENO_CLASICO:.0f} s')

PASO 2.2: RED NEURONAL CON TRANSFER LEARNING (MobileNetV2)

Se parte de **MobileNetV2 preentrenada en ImageNet**, una red ligera pensada para móviles (unos 2.3 M de parámetros en la parte convolucional), y se adapta a las emociones:

* **Entrada:** la cara de 48×48 en grises se aumenta a 96×96 y se replica en 3 canales, porque la red espera imágenes RGB. Se normaliza a [-1, 1], como en su entrenamiento original.
* ***Data augmentation*** (solo al entrenar): volteo horizontal, rotación de ±6 %, zoom de ±10 % y traslación de ±6 %.
* **Cabeza nueva:** *Global Average Pooling* → Dropout 0.3 → Dense 128 (ReLU) → Dropout 0.3 → Dense 7 (softmax).

El entrenamiento se hace en **dos fases**, la receta habitual de *transfer learning*:

1. **Cabeza** (`EPOCAS_CABEZA` épocas): la red base está **congelada** y solo se entrena la cabeza nueva, con tasa de aprendizaje 1e-3.
2. ***Fine-tuning*** (hasta `EPOCAS_FINE` épocas): se **descongelan los últimos bloques** de MobileNetV2 (desde `block_13`) y se entrena todo con una tasa 10 veces menor (1e-4), para ajustar las características de alto nivel a caras sin destruir lo aprendido. Las capas de *batch normalization* se mantienen en modo inferencia.

Callbacks: **parada temprana** (si la accuracy de validación no mejora en 3 épocas se para y se recuperan los mejores pesos), **reducción de la tasa de aprendizaje** en meseta y **guardado del mejor modelo**.

⏱ En CPU (TensorFlow no usa la GPU en Windows) se tarda ~1.6 min por época en la fase 1 y ~5-8 min por época en la fase 2: **entre 45 y 60 minutos** en total. Si el modelo ya está guardado, poner `ENTRENAR_CNN = False` para cargarlo sin volver a entrenar.

In [ ]:
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(SEMILLA)

ENTRENAR_CNN   = True      # False: cargar el modelo ya entrenado de MODELO_CNN
TAM_ENTRADA    = 96
BATCH          = 64
EPOCAS_CABEZA  = 5
EPOCAS_FINE    = 12
CAPA_DESCONGELAR = 'block_13_expand'   # a partir de esta capa se ajusta MobileNetV2 en la fase 2
HISTORIAL_JSON = os.path.join(CARPETA_RESULTADOS, 'historial_cnn.json')


def crear_dataset(X, y, entrenamiento):
    ds = tf.data.Dataset.from_tensor_slices((X[..., None], y))
    if entrenamiento:
        ds = ds.shuffle(len(X), seed=SEMILLA, reshuffle_each_iteration=True)
    return ds.batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32), b)).prefetch(tf.data.AUTOTUNE)


ds_train = crear_dataset(X_train, y_train, True)
ds_val   = crear_dataset(X_val, y_val, False)
ds_test  = crear_dataset(X_test, y_test, False)


def crear_modelo():
    base = keras.applications.MobileNetV2(input_shape=(TAM_ENTRADA, TAM_ENTRADA, 3), include_top=False,
                                          weights='imagenet')
    base.trainable = False
    aumento = keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.06),
                                layers.RandomZoom(0.1), layers.RandomTranslation(0.06, 0.06)], name='aumento')
    entrada = keras.Input((TAM_CARA, TAM_CARA, 1), name='cara_gris')
    h = aumento(entrada)
    h = layers.Resizing(TAM_ENTRADA, TAM_ENTRADA, name='a_96')(h)
    h = layers.Concatenate(name='gris_a_rgb')([h, h, h])
    h = layers.Rescaling(1 / 127.5, offset=-1, name='normalizar')(h)      # [-1, 1], como MobileNetV2
    h = base(h, training=False)                                              # BN siempre en modo inferencia
    h = layers.GlobalAveragePooling2D()(h)
    h = layers.Dropout(0.3)(h)
    h = layers.Dense(128, activation='relu')(h)
    h = layers.Dropout(0.3)(h)
    salida = layers.Dense(N_CLASES, activation='softmax', name='emocion')(h)
    return keras.Model(entrada, salida, name='emociones_mobilenetv2'), base


modelo_cnn, base_cnn = crear_modelo()
modelo_cnn.summary(show_trainable=True)

In [ ]:
if ENTRENAR_CNN:
    t0 = time.time()
    # ---- Fase 1: solo la cabeza
    modelo_cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    h1 = modelo_cnn.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_CABEZA, class_weight=PESOS_CLASE)

    # ---- Fase 2: fine-tuning de los últimos bloques
    base_cnn.trainable = True
    corte = [c.name for c in base_cnn.layers].index(CAPA_DESCONGELAR)
    for capa in base_cnn.layers[:corte]:
        capa.trainable = False
    print(f'Capas de MobileNetV2 ajustadas: {len(base_cnn.layers) - corte} de {len(base_cnn.layers)}')
    modelo_cnn.compile(optimizer=keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
                 keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
                 keras.callbacks.ModelCheckpoint(MODELO_CNN, monitor='val_accuracy', save_best_only=True)]
    h2 = modelo_cnn.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_CABEZA + EPOCAS_FINE,
                        initial_epoch=EPOCAS_CABEZA, class_weight=PESOS_CLASE, callbacks=callbacks)
    TIEMPO_ENTRENO_CNN = time.time() - t0

    historial = {k: h1.history[k] + h2.history[k] for k in ['loss', 'val_loss', 'accuracy', 'val_accuracy']}
    historial['epoca_fine_tuning'] = EPOCAS_CABEZA
    historial['tiempo_s'] = TIEMPO_ENTRENO_CNN
    with open(HISTORIAL_JSON, 'w') as f:
        json.dump(historial, f)
    modelo_cnn = keras.models.load_model(MODELO_CNN)      # mejores pesos según validación
    print(f'Entrenamiento completo en {TIEMPO_ENTRENO_CNN / 60:.1f} min')
else:
    modelo_cnn = keras.models.load_model(MODELO_CNN)
    with open(HISTORIAL_JSON) as f:
        historial = json.load(f)
    TIEMPO_ENTRENO_CNN = historial['tiempo_s']
    print('Modelo cargado de', MODELO_CNN)

In [ ]:
# Curvas de entrenamiento (la línea discontinua marca el inicio del fine-tuning)
epocas = np.arange(1, len(historial['loss']) + 1)
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
for ax, clave, titulo in [(ejes[0], 'loss', 'Pérdida'), (ejes[1], 'accuracy', 'Accuracy')]:
    ax.plot(epocas, historial[clave], 'o-', label='train')
    ax.plot(epocas, historial['val_' + clave], 'o-', label='validación')
    ax.axvline(historial['epoca_fine_tuning'] + 0.5, color='gray', ls='--', label='inicio fine-tuning')
    ax.set_xlabel('Época'); ax.set_title(titulo); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('MobileNetV2 con transfer learning en FER-2013')
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'curvas_cnn.png'), dpi=110)
plt.show()

PASO 2.3: EVALUACIÓN SOBRE TEST

Métricas sobre el conjunto de **test** (PrivateTest), no usado ni para entrenar ni para elegir hiperparámetros:

* **Accuracy:** porcentaje de caras bien clasificadas.
* **F1 macro:** media del F1 de las 7 clases, todas con el mismo peso. Con un dataset desbalanceado es más representativa que la accuracy, porque no se deja dominar por *Alegria*.
* **Matriz de confusión normalizada:** cada fila suma 1 y la diagonal es el recall de cada emoción.

In [ ]:
pred_clasico = modelo_clasico.predict(F_test)
prob_cnn = modelo_cnn.predict(ds_test, verbose=0)
pred_cnn = prob_cnn.argmax(axis=1)

evaluar('HOG+LBP + SVM', y_test, pred_clasico)
evaluar('MobileNetV2 (TL)', y_test, pred_cnn)

fig, ejes = plt.subplots(1, 2, figsize=(15, 6))
dibujar_matriz(ejes[0], RESULTADOS['HOG+LBP + SVM']['matriz'], 'HOG+LBP + SVM')
dibujar_matriz(ejes[1], RESULTADOS['MobileNetV2 (TL)']['matriz'], 'MobileNetV2 (transfer learning)')
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'matrices_confusion.png'), dpi=110)
plt.show()

In [ ]:
# Ejemplos de test con la predicción de la red (verde = acierto, rojo = fallo)
idx = np.random.choice(len(X_test), 18, replace=False)
plt.figure(figsize=(15, 6))
for k, i in enumerate(idx):
    plt.subplot(3, 6, k + 1)
    plt.imshow(X_test[i], cmap='gray'); plt.axis('off')
    ok = pred_cnn[i] == y_test[i]
    plt.title(f'{EMOCIONES[pred_cnn[i]]} ({prob_cnn[i].max():.0%})\nreal: {EMOCIONES[y_test[i]]}',
              color='green' if ok else 'red', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'ejemplos_prediccion.png'), dpi=100)
plt.show()

PASO 2.4 (EXTRA): COMPARATIVA ENTRE EL MODELO CLÁSICO Y LA RED

Además de las métricas, se comparan el **tiempo de entrenamiento**, el **tiempo de inferencia por cara** (lo que importa en el prototipo en tiempo real, medido cara a cara tras un calentamiento) y el **tamaño** del modelo.

In [ ]:
import joblib

joblib.dump(modelo_clasico, MODELO_CLASICO)

inferir_cnn = tf.function(lambda x: modelo_cnn(x, training=False),
                          input_signature=[tf.TensorSpec([None, TAM_CARA, TAM_CARA, 1], tf.float32)])


def medir_ms(funcion, n=200):
    for i in range(5):                                   # calentamiento
        funcion(X_test[i])
    t0 = time.perf_counter()
    for i in range(n):
        funcion(X_test[i])
    return (time.perf_counter() - t0) / n * 1000


ms_clasico = medir_ms(lambda c: modelo_clasico.predict(caracteristicas_clasicas(c[None])))
ms_cnn = medir_ms(lambda c: inferir_cnn(c[None, ..., None].astype(np.float32)).numpy())

RESULTADOS['HOG+LBP + SVM'].update({'entreno_min': TIEMPO_ENTRENO_CLASICO / 60, 'inferencia_ms': ms_clasico,
                                    'tamano_MB': os.path.getsize(MODELO_CLASICO) / 1e6})
RESULTADOS['MobileNetV2 (TL)'].update({'entreno_min': TIEMPO_ENTRENO_CNN / 60, 'inferencia_ms': ms_cnn,
                                       'tamano_MB': os.path.getsize(MODELO_CNN) / 1e6})

tabla = pd.DataFrame({n: {k: r[k] for k in ['accuracy', 'f1_macro', 'f1_ponderado', 'entreno_min',
                                            'inferencia_ms', 'tamano_MB']}
                      for n, r in RESULTADOS.items()}).T
tabla.to_csv(os.path.join(CARPETA_RESULTADOS, 'comparativa_modelos.csv'))
display(tabla.style.format({'accuracy': '{:.3f}', 'f1_macro': '{:.3f}', 'f1_ponderado': '{:.3f}',
                            'entreno_min': '{:.1f}', 'inferencia_ms': '{:.1f}', 'tamano_MB': '{:.1f}'}))

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
nombres = ['HOG+LBP + SVM', 'MobileNetV2 (TL)']
colores = ['#4C72B0', '#DD8452']

x = np.arange(N_CLASES)
for k, n in enumerate(nombres):
    ejes[0].bar(x + (k - 0.5) * 0.4, RESULTADOS[n]['f1_por_clase'], 0.4, label=n, color=colores[k])
ejes[0].set_xticks(x, EMOCIONES, rotation=30); ejes[0].set_ylim(0, 1)
ejes[0].set_title('F1 por emoción (test)'); ejes[0].legend()

b = ejes[1].bar(nombres, [RESULTADOS[n]['f1_macro'] for n in nombres], color=colores)
ejes[1].bar_label(b, fmt='%.3f'); ejes[1].set_ylim(0, 1); ejes[1].set_title('F1 macro (test)')

b = ejes[2].bar(nombres, [RESULTADOS[n]['inferencia_ms'] for n in nombres], color=colores)
ejes[2].bar_label(b, fmt='%.1f ms'); ejes[2].set_title('Tiempo de inferencia por cara')

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'comparativa_modelos.png'), dpi=110)
plt.show()

PASO 2.5 (EXTRA): COMPARATIVA CON `DeepFace.analyze`

Se pasan `N_DEEPFACE` caras de test por el modelo de emociones de deepface (con `detector_backend='skip'`, porque las caras ya están recortadas) y se compara con nuestros dos modelos **sobre las mismas imágenes**.

Hay que tener en cuenta que **el modelo de emociones de deepface también se entrenó con FER-2013**, así que juega en casa. Aun así sirve como referencia de un modelo ya hecho.

In [ ]:
from deepface import DeepFace

N_DEEPFACE = 500
A_INDICE = {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4, 'surprise': 5, 'neutral': 6}
sub = np.random.RandomState(SEMILLA).choice(len(X_test), N_DEEPFACE, replace=False)

pred_df, tiempos_df = [], []
for i in sub:
    img_bgr = cv2.cvtColor(X_test[i], cv2.COLOR_GRAY2BGR)
    t0 = time.perf_counter()
    r = DeepFace.analyze(img_path=img_bgr, actions=['emotion'], detector_backend='skip',
                         enforce_detection=False, silent=True)
    tiempos_df.append(time.perf_counter() - t0)
    pred_df.append(A_INDICE[r[0]['dominant_emotion']])
pred_df = np.array(pred_df)

print(f'--- Las mismas {N_DEEPFACE} caras de test ---')
comp_df = {}
for nombre, pred in [('DeepFace.analyze', pred_df), ('HOG+LBP + SVM', pred_clasico[sub]),
                     ('MobileNetV2 (TL)', pred_cnn[sub])]:
    comp_df[nombre] = {'accuracy': accuracy_score(y_test[sub], pred),
                       'f1_macro': f1_score(y_test[sub], pred, average='macro')}
comp_df['DeepFace.analyze']['inferencia_ms'] = np.median(tiempos_df[5:]) * 1000
comp_df['HOG+LBP + SVM']['inferencia_ms'] = RESULTADOS['HOG+LBP + SVM']['inferencia_ms']
comp_df['MobileNetV2 (TL)']['inferencia_ms'] = RESULTADOS['MobileNetV2 (TL)']['inferencia_ms']
comp_df = pd.DataFrame(comp_df).T
comp_df.to_csv(os.path.join(CARPETA_RESULTADOS, 'comparativa_deepface.csv'))
display(comp_df.style.format({'accuracy': '{:.3f}', 'f1_macro': '{:.3f}', 'inferencia_ms': '{:.1f}'}))

fig, ax = plt.subplots(figsize=(7, 5))
dibujar_matriz(ax, confusion_matrix(y_test[sub], pred_df, labels=range(N_CLASES)), 'DeepFace.analyze')
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'matriz_deepface.png'), dpi=110)
plt.show()

PASO 2.6: MODELO ELEGIDO

Los dos modelos quedan guardados en `../modelos/`:

* `emociones_mobilenetv2.keras`: la red (mejores pesos según validación).
* `emociones_hog_lbp_svm.joblib`: el modelo clásico.

**Para el prototipo se usa la red MobileNetV2.** Las redes con *transfer learning* suelen superar con claridad a los descriptores clásicos en FER-2013, y su tiempo de inferencia en CPU es suficiente para la webcam.

*TODO tras ejecutar: confirmar esta elección con los números de los pasos 2.3 a 2.5 (accuracy, F1 macro, emociones en las que más mejora y tiempo por cara).*

---
# PARTE 3: PROTOTIPO EN TIEMPO REAL — ESPEJO DE EMOCIONES

> Para usar solo el prototipo, sin cargar el dataset ni volver a entrenar, basta con ejecutar: **paquetes**, **configuración**, las **dos celdas del paso 1.3** (funciones de landmarks y prueba con imágenes) y esta Parte 3. El modelo se carga de `../modelos/emociones_mobilenetv2.keras`.

PASO 3.1: CARGAR EL DETECTOR DE LANDMARKS Y EL MODELO ENTRENADO

* **Detección y landmarks:** MediaPipe Face Landmarker en modo `VIDEO` (síncrono), hasta `NUM_CARAS` caras a la vez.
* **Clasificación:** la red MobileNetV2 entrenada en la Parte 2, envuelta en una `tf.function` para que la inferencia por fotograma sea rápida.

In [ ]:
import tensorflow as tf
import keras

NUM_CARAS = 2
CAMARA    = 0

modelo_emociones = keras.models.load_model(MODELO_CNN)
inferir = tf.function(lambda x: modelo_emociones(x, training=False),
                      input_signature=[tf.TensorSpec([None, TAM_CARA, TAM_CARA, 1], tf.float32)])
inferir(np.zeros((1, TAM_CARA, TAM_CARA, 1), np.float32))      # calentamiento

opciones_video = FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=FACE_LANDMARKER),
                                       running_mode=VisionRunningMode.VIDEO, num_faces=NUM_CARAS)
print('Modelo y opciones del detector listos')

PASO 3.2: FUNCIONES DEL PROTOTIPO

* **`predecir_emociones`**: recorta y alinea cada cara (paso 1.3) y clasifica todas en un solo lote.
* **Suavizado temporal:** la predicción de un único fotograma oscila mucho, así que se usa una **media móvil exponencial** de las probabilidades (`SUAVIZADO`). Esto hace el prototipo más **robusto** y evita que los efectos parpadeen.
* **Reacciones**, ancladas a los landmarks y escaladas con la distancia entre ojos para que funcionen a cualquier distancia de la cámara:

| Emoción | Reacción |
|---|---|
| Alegria | Rubor en las mejillas y destellos girando alrededor de la cabeza |
| Tristeza | Lágrimas que caen desde los párpados inferiores |
| Enfado | Cara teñida de rojo y vapor saliendo de la cabeza |
| Sorpresa | Signos de exclamación que rebotan sobre la cabeza |
| Miedo | Tinte morado y contorno de la cara temblando |
| Asco | Tinte verde sobre la cara |
| Neutral | Sin efecto |

* **Panel** con la probabilidad de cada emoción y **modo juego** *"Imita la emoción"*.

In [ ]:
SUAVIZADO   = 0.35    # peso del fotograma nuevo en la media móvil (más bajo = más estable, más lento)
UMBRAL_EFECTO = 0.40  # probabilidad mínima para lanzar el efecto de una emoción


def predecir_emociones(frame, caras):
    """Probabilidades (n_caras, 7) para las caras detectadas (lista de arrays de landmarks)."""
    if not caras:
        return np.zeros((0, N_CLASES), np.float32)
    lote = np.stack([recortar_cara_alineada(frame, pts) for pts in caras]).astype(np.float32)[..., None]
    return inferir(lote).numpy()


class Suavizador:
    """Media móvil exponencial de las probabilidades de cada cara (por orden de detección)."""
    def __init__(self, alfa=SUAVIZADO):
        self.alfa, self.estado = alfa, []

    def actualizar(self, probs):
        if len(probs) != len(self.estado):          # cambia el nº de caras: se reinicia
            self.estado = [p.copy() for p in probs]
        else:
            self.estado = [self.alfa * p + (1 - self.alfa) * e for p, e in zip(probs, self.estado)]
        return self.estado


# ---------------------------------------------------------------- utilidades de dibujo
def tinte(frame, puntos, color, alfa=0.35):
    """Colorea la zona de la cara (dentro del óvalo facial) con transparencia."""
    capa = frame.copy()
    cv2.fillPoly(capa, [puntos[OVALO].astype(np.int32)], color)
    cv2.addWeighted(capa, alfa, frame, 1 - alfa, 0, dst=frame)


def estrella(frame, centro, radio, color, angulo=0):
    pts = []
    for k in range(10):
        r = radio if k % 2 == 0 else radio * 0.45
        a = angulo + k * np.pi / 5
        pts.append((centro[0] + r * np.cos(a), centro[1] + r * np.sin(a)))
    cv2.fillPoly(frame, [np.array(pts, np.int32)], color, lineType=cv2.LINE_AA)


def escala_cara(puntos):
    return float(np.linalg.norm(puntos[OJO_DER].mean(0) - puntos[OJO_IZQ].mean(0)))


# ---------------------------------------------------------------- reacciones
def efecto_alegria(frame, p, t):
    s = escala_cara(p)
    capa = frame.copy()
    for i in (MEJILLA_IZQ, MEJILLA_DER):
        cv2.ellipse(capa, tuple(p[i].astype(int)), (int(s * 0.35), int(s * 0.22)), 0, 0, 360, (150, 120, 255), -1)
    cv2.addWeighted(capa, 0.45, frame, 0.55, 0, dst=frame)
    centro = (p[FRENTE] + p[BARBILLA]) / 2
    radio = np.linalg.norm(p[BARBILLA] - p[FRENTE]) * 0.75
    for k in range(6):
        a = t * 1.5 + k * np.pi / 3
        estrella(frame, centro + radio * np.array([np.cos(a), np.sin(a)]), s * 0.18, (0, 230, 255), a)


def efecto_tristeza(frame, p, t):
    s = escala_cara(p)
    for i, desfase in ((PARPADO_IZQ, 0.0), (PARPADO_DER, 0.5)):
        caida = ((t * 0.8 + desfase) % 1.0) * s * 1.4
        x, y = p[i]
        cv2.ellipse(frame, (int(x), int(y + s * 0.15 + caida)), (int(s * 0.07), int(s * 0.12)),
                    0, 0, 360, (255, 170, 60), -1, cv2.LINE_AA)


def efecto_enfado(frame, p, t):
    tinte(frame, p, (0, 0, 255), 0.30)
    s = escala_cara(p)
    x0, y0 = p[FRENTE]
    for k, dx in enumerate((-0.6, 0.0, 0.6)):
        pts = [(x0 + dx * s + np.sin(t * 8 + j + k) * s * 0.08, y0 - s * 0.3 - j * s * 0.12) for j in range(8)]
        cv2.polylines(frame, [np.array(pts, np.int32)], False, (220, 220, 220), max(2, int(s * 0.06)), cv2.LINE_AA)


def efecto_sorpresa(frame, p, t):
    s = escala_cara(p)
    salto = abs(np.sin(t * 6)) * s * 0.3
    x0, y0 = p[FRENTE]
    for dx in (-0.5, 0.5):
        cv2.putText(frame, '!', (int(x0 + dx * s - s * 0.12), int(y0 - s * 0.3 - salto)),
                    cv2.FONT_HERSHEY_DUPLEX, s / 40, (0, 140, 255), max(2, int(s / 12)), cv2.LINE_AA)


def efecto_miedo(frame, p, t):
    tinte(frame, p, (160, 40, 140), 0.30)
    temblor = np.array([np.sin(t * 40), np.cos(t * 35)]) * escala_cara(p) * 0.05
    cv2.polylines(frame, [(p[OVALO] + temblor).astype(np.int32)], True, (200, 80, 200), 3, cv2.LINE_AA)


def efecto_asco(frame, p, t):
    tinte(frame, p, (40, 200, 40), 0.35)


EFECTOS = {'Alegria': efecto_alegria, 'Tristeza': efecto_tristeza, 'Enfado': efecto_enfado,
           'Sorpresa': efecto_sorpresa, 'Miedo': efecto_miedo, 'Asco': efecto_asco}


def dibujar_panel(frame, probs, x=10, y=10):
    """Barras con la probabilidad de cada emoción para la primera cara."""
    capa = frame.copy()
    cv2.rectangle(capa, (x, y), (x + 260, y + 30 + 26 * N_CLASES), (0, 0, 0), -1)
    cv2.addWeighted(capa, 0.55, frame, 0.45, 0, dst=frame)
    for k, (emocion, pr) in enumerate(zip(EMOCIONES, probs)):
        yy = y + 30 + 26 * k
        cv2.putText(frame, emocion, (x + 8, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.rectangle(frame, (x + 100, yy - 14), (x + 100 + int(140 * pr), yy), COLORES[emocion], -1)
        cv2.putText(frame, f'{pr:.0%}', (x + 205, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)


class JuegoImitar:
    """Modo juego: pide una emoción y suma un punto si se mantiene SEGUNDOS_OBJETIVO seguidos."""
    OBJETIVOS = ['Alegria', 'Sorpresa', 'Enfado', 'Tristeza', 'Neutral']   # las que el modelo reconoce mejor

    def __init__(self, segundos_objetivo=1.0, limite=10.0, umbral=0.45):
        self.seg, self.limite, self.umbral = segundos_objetivo, limite, umbral
        self.puntos, self.rondas = 0, 0
        self.nuevo_objetivo(time.time())

    def nuevo_objetivo(self, ahora, anterior=None):
        opciones = [e for e in self.OBJETIVOS if e != anterior]
        self.objetivo = opciones[np.random.randint(len(opciones))]
        self.inicio, self.mantenido_desde = ahora, None

    def actualizar(self, probs, ahora):
        prob = probs[EMOCIONES.index(self.objetivo)] if probs is not None else 0.0
        if prob >= self.umbral:
            self.mantenido_desde = self.mantenido_desde or ahora
            if ahora - self.mantenido_desde >= self.seg:
                self.puntos += 1; self.rondas += 1
                self.nuevo_objetivo(ahora, self.objetivo)
                return True
        else:
            self.mantenido_desde = None
        if ahora - self.inicio > self.limite:
            self.rondas += 1
            self.nuevo_objetivo(ahora, self.objetivo)
        return False

    def dibujar(self, frame, ahora):
        h, w = frame.shape[:2]
        restante = max(0.0, self.limite - (ahora - self.inicio))
        texto = f'Pon cara de: {self.objetivo.upper()}'
        cv2.putText(frame, texto, (w // 2 - 230, 45), cv2.FONT_HERSHEY_DUPLEX, 1.1, (0, 0, 0), 5, cv2.LINE_AA)
        cv2.putText(frame, texto, (w // 2 - 230, 45), cv2.FONT_HERSHEY_DUPLEX, 1.1, COLORES[self.objetivo], 2, cv2.LINE_AA)
        cv2.putText(frame, f'Puntos: {self.puntos}/{self.rondas}   Tiempo: {restante:.0f}s', (w // 2 - 170, 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        if self.mantenido_desde:
            progreso = min(1.0, (ahora - self.mantenido_desde) / self.seg)
            cv2.rectangle(frame, (w // 2 - 150, 100), (w // 2 - 150 + int(300 * progreso), 115), COLORES[self.objetivo], -1)
            cv2.rectangle(frame, (w // 2 - 150, 100), (w // 2 + 150, 115), (255, 255, 255), 2)


def procesar_fotograma(frame, caras, probs, t, mostrar_malla=False):
    """Dibuja las reacciones, la etiqueta de cada cara y la malla de landmarks si se pide."""
    for pts, pr in zip(caras, probs):
        emocion = EMOCIONES[int(np.argmax(pr))]
        if pr.max() >= UMBRAL_EFECTO and emocion in EFECTOS:
            EFECTOS[emocion](frame, pts, t)
        if mostrar_malla:
            for x, y in pts[::3].astype(int):
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)
        x1, y1 = pts.min(axis=0).astype(int)
        x2, y2 = pts.max(axis=0).astype(int)
        cv2.putText(frame, f'{emocion} {pr.max():.0%}', (x1, y2 + 30), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                    (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(frame, f'{emocion} {pr.max():.0%}', (x1, y2 + 30), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                    COLORES[emocion], 2, cv2.LINE_AA)

PASO 3.3: PRUEBA SOBRE LAS IMÁGENES DE `../img/test`

Antes de abrir la webcam se comprueba el flujo completo (landmarks → recorte alineado → modelo → reacción) con imágenes fijas.

In [ ]:
plt.figure(figsize=(16, 5))
with FaceLandmarker.create_from_options(opciones_imagen) as detector_img:
    for k, nombre in enumerate(imagenes_test):
        img = cv2.imread(os.path.join(CARPETA_TEST_IMG, nombre))
        res = detector_img.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
        caras = landmarks_a_pixeles(res, img.shape[1], img.shape[0])
        probs = predecir_emociones(img, caras)
        procesar_fotograma(img, caras, probs, t=0.3)
        if len(probs):
            dibujar_panel(img, probs[0])
        plt.subplot(1, len(imagenes_test), k + 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(nombre); plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, 'prueba_imagenes.png'), dpi=90)
plt.show()

PASO 3.4: ESPEJO DE EMOCIONES CON LA WEBCAM

Teclas:

| Tecla | Acción |
|---|---|
| `j` | Activa o desactiva el modo juego *"Imita la emoción"* |
| `m` | Muestra u oculta la malla de landmarks |
| `g` | Empieza o termina la grabación de un clip en `../resultados/tarea1/` (para el gif de la entrega) |
| `Esc` | Salir |

In [ ]:
cap = cv2.VideoCapture(CAMARA)
suavizador = Suavizador()
juego, mostrar_malla = None, False
grabador, n_clip = None, 0
t_inicio, t_prev = time.time(), time.time()

with FaceLandmarker.create_from_options(opciones_video) as detector:
    while True:
        ok, frame = cap.read()
        if not ok:
            print('No se pudo leer de la cámara'); break
        frame = cv2.flip(frame, 1)                                  # efecto espejo
        ahora = time.time()
        t = ahora - t_inicio

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(t * 1000))
        caras = landmarks_a_pixeles(res, frame.shape[1], frame.shape[0])
        probs = suavizador.actualizar(predecir_emociones(frame, caras))

        procesar_fotograma(frame, caras, probs, t, mostrar_malla)
        if probs:
            dibujar_panel(frame, probs[0])
        else:
            cv2.putText(frame, 'No se detecta ninguna cara', (20, frame.shape[0] - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)

        if juego:
            if juego.actualizar(probs[0] if probs else None, ahora):
                cv2.rectangle(frame, (0, 0), (frame.shape[1] - 1, frame.shape[0] - 1), (0, 255, 0), 20)
            juego.dibujar(frame, ahora)

        fps = 1.0 / max(ahora - t_prev, 1e-6); t_prev = ahora
        cv2.putText(frame, f'{fps:.0f} FPS', (frame.shape[1] - 110, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        if grabador is not None:
            grabador.write(frame)
            cv2.circle(frame, (frame.shape[1] - 25, 25), 10, (0, 0, 255), -1)

        cv2.imshow('Espejo de emociones (j: juego, m: malla, g: grabar, Esc: salir)', frame)
        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27:
            break
        elif tecla == ord('j'):
            juego = None if juego else JuegoImitar()
        elif tecla == ord('m'):
            mostrar_malla = not mostrar_malla
        elif tecla == ord('g'):
            if grabador is None:
                ruta = os.path.join(CARPETA_RESULTADOS, f'clip_tarea1_{n_clip}.mp4'); n_clip += 1
                grabador = cv2.VideoWriter(ruta, cv2.VideoWriter_fourcc(*'mp4v'), 20, (frame.shape[1], frame.shape[0]))
                print('Grabando en', ruta)
            else:
                grabador.release(); grabador = None
                print('Grabación terminada')

cap.release()
if grabador is not None:
    grabador.release()
cv2.destroyAllWindows()
if juego:
    print(f'Puntuación final: {juego.puntos} de {juego.rondas}')